In [1]:
!pip install -U autotrain-advanced

  Using cached huggingface_hub-0.27.0-py3-none-any.whl.metadata (13 kB)
Using cached huggingface_hub-0.27.0-py3-none-any.whl (450 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.30.2
    Uninstalling huggingface-hub-0.30.2:
      Successfully uninstalled huggingface-hub-0.30.2


In [2]:
!pip install transformers datasets

In [3]:
from google.colab import files
uploaded = files.upload()

Saving twittersentiment_1_1000_enriched.csv to twittersentiment_1_1000_enriched (1).csv


In [4]:
import pandas as pd
from datasets import Dataset

# Load CSV
df = pd.read_csv("/content/twittersentiment_1_1000_enriched.csv")

# Only keep relevant columns
df = df[["text", "Mistral_label"]]
df = df.dropna()

# Define label mapping
label_map = {
    "negative": 0,
    "positive": 2,
    "neutral": 1  # Include this if you have a 3-class problem
}

# Apply mapping
df["Mistral_label"] = df["Mistral_label"].map(label_map)

# Drop rows where label mapping failed
df = df.dropna(subset=["Mistral_label"])

# Convert to integer type
df["Mistral_label"] = df["Mistral_label"].astype(int)
df

,text,Mistral_label
0,Sooo SAD I will miss you here in San Diego!!!,0
1,my boss is bullying me...,2
2,what interview! leave me alone,0
3,"Sons of ****, why couldn`t they put them on t...",0
4,http://www.dothebouncy.com/smf - some shameles...,2
...,...,...
993,"I am twittering, LIKE A BOSS. Thanks Savvv",1
994,my sleep pattern is screwed i need to try and...,0
995,Since the demise of Woolworths it isn`t easy ...,0
996,may the fourth be with you! happy star wars day,2


In [5]:
dataset = Dataset.from_pandas(df)
dataset

Dataset({
    features: ['text', 'Mistral_label', '__index_level_0__'],
    num_rows: 994
})

In [6]:
from transformers import AutoTokenizer

model_name = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.rename_column("Mistral_label", "labels")
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/994 [00:00<?, ? examples/s]

In [7]:
tokenized_dataset


Dataset({
    features: ['text', 'labels', '__index_level_0__', 'input_ids', 'attention_mask'],
    num_rows: 994
})

In [24]:
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Optional: Confirm GPU is available
print("Using GPU:", torch.cuda.is_available())

# Load model with number of labels
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    push_to_hub=True,  # change to True if pushing after training
    hub_model_id="RajeevanL/Distill_Roberta",
    report_to="none"    # set to "wandb" or "tensorboard" if using logging
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,  # use a validation split for better generalization
    tokenizer=tokenizer,
)

# Train
trainer.train()


Using GPU: True


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-24-890314ac0601>:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,0.882800,0.897331
2,0.832500,0.695811
3,0.618500,0.525117


TrainOutput(global_step=189, training_loss=0.8287791877827316, metrics={'train_runtime': 166.2784, 'train_samples_per_second': 17.934, 'train_steps_per_second': 1.137, 'total_flos': 784604211664896.0, 'train_loss': 0.8287791877827316, 'epoch': 3.0})

In [9]:
# !pip install huggingface_hub --upgrade
from huggingface_hub import HfApi, login

  Using cached huggingface_hub-0.30.2-py3-none-any.whl.metadata (13 kB)
Using cached huggingface_hub-0.30.2-py3-none-any.whl (481 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.27.0
    Uninstalling huggingface-hub-0.27.0:
      Successfully uninstalled huggingface-hub-0.27.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autotrain-advanced 0.8.36 requires huggingface-hub==0.27.0, but you have huggingface-hub 0.30.2 which is incompatible.


In [ ]:
login("token")
